In [ ]:
%matplotlib widget

# PPO training

In [ ]:
import jax
from flock.env import EnvConfig, run_episode
from flock.env.rules import PredatorPrey
from flock.env.viz import animate, plot_stats
from flock.train.ppo import PPOConfig, Trainee, PredatorReward, PreyReward, make_policy, train

In [ ]:
rules = PredatorPrey(
    n_predators_init=5,
    n_prey_init=10,
    n_predators_max=10,
    n_prey_max=20,
    catch_radius=0.3,
    k_teammates=4,
    k_opponents=5,
)
env_cfg = EnvConfig(
    max_steps=400,
    dt=0.1,
)

key = jax.random.key(42)
k1, k2 = jax.random.split(key)
pred_policy = make_policy(rules.teams[0], key=k1)
prey_policy = make_policy(rules.teams[1], key=k2)

## Before training

In [ ]:
sim_before = run_episode(env_cfg, rules, jax.random.key(0), (pred_policy, prey_policy))
_ = animate(sim_before)

In [ ]:
_ = plot_stats(sim_before)

## Self-play

In [ ]:
cfg = PPOConfig(
    n_iters=236,
)

trained_policies = train(
    env_cfg, rules,
    policies=(pred_policy, prey_policy),
    trainees=(
        Trainee(team_idx=0, reward_fn=PredatorReward()),
        Trainee(team_idx=1, reward_fn=PreyReward()),
    ),
    key=jax.random.key(1),
    cfg=cfg,
)

## After training

In [ ]:
sim_after = run_episode(env_cfg, rules, jax.random.key(0), trained_policies)
_ = animate(sim_after)

In [ ]:
_ = plot_stats(sim_after)

In [ ]:
from flock.env.serialize import save
save(sim_after, "selfplay")
plot = animate(sim_after)
plot.save("selfplay.mp4", writer="ffmpeg", fps=30)